# Understanding RAG

In [62]:
from openai import OpenAI

In [63]:
from google.colab import userdata
api_key = userdata.get("OPENAI_API_KEY")

client = OpenAI(api_key=api_key)

In [64]:
# Step 1: Create Embeddings

response = client.embeddings.create(
    model="text-embedding-3-small",
    input="Machine learning is amazing"
)

embedding = response.data[0].embedding
print(len(embedding))  # vector length

1536


In [65]:
# Step 2: Store (Simple Python List)

documents = [
    "Machine learning is a subset of AI",
    "Deep learning uses neural networks",
    "Python is a programming language"
]

doc_embeddings = []

for doc in documents:
    emb = client.embeddings.create(
        model="text-embedding-3-small",
        input=doc
    ).data[0].embedding

    doc_embeddings.append((doc, emb))

In [66]:
doc_embeddings[0]

('Machine learning is a subset of AI',
 [-0.023230766877532005,
  -0.05649999529123306,
  -0.00947596039623022,
  -0.019980767741799355,
  0.0504615344107151,
  -0.009980768896639347,
  0.017519230023026466,
  0.04642307385802269,
  -0.027230767533183098,
  0.020403845235705376,
  -0.032596152275800705,
  -0.03753845766186714,
  -0.03488461300730705,
  0.004406249616295099,
  0.0023713940754532814,
  -0.0020276440773159266,
  -0.01064423006027937,
  0.008778845891356468,
  0.041961535811424255,
  -0.011028844863176346,
  0.005399038083851337,
  0.038576919585466385,
  0.022346151992678642,
  0.0016057691536843777,
  0.019769228994846344,
  -0.03492307290434837,
  0.01894230581820011,
  0.0376538448035717,
  -0.005913461092859507,
  0.014759614132344723,
  0.03290384262800217,
  -0.016932690516114235,
  -0.06353845447301865,
  -0.045192304998636246,
  0.009855768643319607,
  0.044653844088315964,
  0.025615382939577103,
  -0.0019483171636238694,
  0.0032836536411195993,
  0.042038459330

In [67]:
# Step 3: Search (Similarity)
# We use cosine similarity

import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

query = "What is AI?"

query_emb = client.embeddings.create(
    model="text-embedding-3-small",
    input=query
).data[0].embedding

# Find best match
best_doc = None
best_score = -1

for doc, emb in doc_embeddings:
    score = cosine_similarity(query_emb, emb)
    if score > best_score:
        best_score = score
        best_doc = doc

print("Best match:", best_doc)

Best match: Machine learning is a subset of AI


cosine similarity = (A · B) / (|A| × |B|)

Range:

+1 → very similar

0 → unrelated

-1 → opposite

#RAG: Implementation

In [68]:
from google.colab import userdata
api_key = userdata.get("OPENAI_API_KEY")

client = OpenAI(api_key=api_key)

## Load Documents

In [69]:
document = """
Machine learning is a field of artificial intelligence that focuses on building systems that can learn from data and improve over time without being explicitly programmed.

One of the most fundamental concepts in machine learning is supervised learning. In supervised learning, the model is trained using labeled data.

Another important paradigm is unsupervised learning. Unlike supervised learning, this approach works with unlabeled data.

Reinforcement learning is another branch where an agent learns by interacting with an environment.

A core algorithm used in many machine learning models is gradient descent. Gradient descent is an optimization algorithm used to minimize a loss function.

There are several advantages of gradient descent. It is simple to implement and computationally efficient.

Deep learning is a subset of machine learning that uses neural networks with multiple layers.

Transformers have become the dominant architecture in natural language processing.

Machine learning systems are widely used in real-world applications such as recommendation systems and fraud detection.

Despite its power, machine learning has challenges like overfitting and bias.

To build effective machine learning systems, it is important to follow a proper workflow.

In modern AI systems, retrieval-augmented generation (RAG) combines machine learning with information retrieval.

Chunking is an important step in RAG pipelines.

Overall, machine learning and RAG systems are transforming how intelligent applications are built.
"""

## Chunking

In [70]:
def chunk_text(text, chunk_size=60, overlap=10):
    words = text.split()
    chunks = []

    i = 0
    while i < len(words):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap

    return chunks

In [71]:
chunks = chunk_text(document, chunk_size=60, overlap=10)

print("Number of chunks:", len(chunks))
print("\nFirst chunk:\n", chunks[0])

Number of chunks: 5

First chunk:
 Machine learning is a field of artificial intelligence that focuses on building systems that can learn from data and improve over time without being explicitly programmed. One of the most fundamental concepts in machine learning is supervised learning. In supervised learning, the model is trained using labeled data. Another important paradigm is unsupervised learning. Unlike supervised learning, this approach works


## Create Embeddings

In [72]:
def embed(text):
    return client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    ).data[0].embedding

chunk_embeddings = [(chunk, embed(chunk)) for chunk in chunks]

In [73]:
print(chunk_embeddings[0])

('Machine learning is a field of artificial intelligence that focuses on building systems that can learn from data and improve over time without being explicitly programmed. One of the most fundamental concepts in machine learning is supervised learning. In supervised learning, the model is trained using labeled data. Another important paradigm is unsupervised learning. Unlike supervised learning, this approach works', [-0.0233917236328125, -0.01499176025390625, 0.010589599609375, -0.0028591156005859375, 0.041107177734375, -0.00012540817260742188, 0.020416259765625, 0.0127105712890625, -0.0205078125, 0.06524658203125, 0.018951416015625, -0.0199737548828125, -0.052886962890625, -0.0026493072509765625, 0.006244659423828125, 0.01605224609375, 0.0013265609741210938, -0.00879669189453125, 0.058746337890625, 0.00981903076171875, 0.005252838134765625, 0.0438232421875, 0.033599853515625, 0.005290985107421875, 0.01152801513671875, -0.033935546875, -0.01172637939453125, 0.05072021484375, -0.0259

## Vector Search (Cosine Similarity)

In [74]:
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def retrieve(query, k=5):
    q_emb = embed(query)
    scored = [(chunk, cosine_similarity(q_emb, emb)) for chunk, emb in chunk_embeddings]
    return sorted(scored, key=lambda x: x[1], reverse=True)[:k]

## Re-ranking with cross-encoder
(query + chunk) → deep model → relevance score

In [ ]:
from sentence_transformers import CrossEncoder

reranker_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2') # model from Hugging Face

def rerank(query, retrieved_chunks):
    # retrieved_chunks = [(chunk, similarity_score)]

    chunks = [chunk for chunk, _ in retrieved_chunks]

    pairs = [[query, chunk] for chunk in chunks]

    scores = reranker_model.predict(pairs)

    reranked = list(zip(chunks, scores))

    return sorted(reranked, key=lambda x: x[1], reverse=True)

In [76]:
query = "What are advantages of gradient descent?"

retrieved = retrieve(query, k=5)

print("Before reranking:")
for c, s in retrieved:
    print(f"{round(s,3)} → {c[:60]}")

reranked = rerank(query, retrieved)

print("\nAfter reranking:")
for c, s in reranked:
    print(f"{round(s,3)} → {c[:60]}")

Before reranking:
0.663 → function. There are several advantages of gradient descent. 
0.437 → paradigm is unsupervised learning. Unlike supervised learnin
0.268 → such as recommendation systems and fraud detection. Despite 
0.262 → in RAG pipelines. Overall, machine learning and RAG systems 
0.213 → Machine learning is a field of artificial intelligence that 

After reranking:
8.829999923706055 → function. There are several advantages of gradient descent. 
6.294000148773193 → paradigm is unsupervised learning. Unlike supervised learnin
-11.196000099182129 → such as recommendation systems and fraud detection. Despite 
-11.218000411987305 → Machine learning is a field of artificial intelligence that 
-11.390999794006348 → in RAG pipelines. Overall, machine learning and RAG systems 


## RAG in action

In [77]:
top_chunks = reranked[:2]

context = "\n".join([c for c, _ in top_chunks])

prompt = f"""
Answer using only the context below:

{context}

Question: {query}
"""

response = client.responses.create(
    model="gpt-4o-mini",
    input=prompt
)

print(response.output_text)

Gradient descent is simple to implement and computationally efficient.
